# Simulation Failure Analysis (v3)

Classifies every simulated case (denominator = the canonical **v3** 320-case set)
into one outcome and breaks failures down by type and by patient persona. Uses the
canonical simulation run; run-to-run variation in failure rates across the 5 repeats
is reported separately in the multi-seed variance study
(`data/processed/variance_study_report.md`, sanity panel).

**Outcomes**
- `clean_complete` — env agent marked `session_complete` after the min-turn guard, no exception.
- `early_termination` — env agent hard-stopped the session (a severe persona break by the VP, not an educator failure).
- `max_turns` — hit the turn cap without `session_complete`.
- `error_*` — an exception aborted the run: `connection` (vLLM server unreachable — infrastructure),
  `context_length` (400 BadRequest), `json_parse`, `other`.

**NOTE:** `error_connection` is an infrastructure artifact, not model behavior. The
behavior-failure panels (`early_termination` + `max_turns`) are the model-attributable signal.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_ROOT  = PROJECT_ROOT / "data/processed/output"
FIG_DIR      = PROJECT_ROOT / "figures"
OUT_DIR      = PROJECT_ROOT / "outputs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (display name, output-directory name) -- only models with simulation_results.jsonl
MODELS = [
    ("Llama-3.3-70B-Instruct", "Llama-3.3-70B-Instruct"),
    ("Qwen3-4B",               "Qwen3-4B"),
    ("Qwen3-32B",              "Qwen3-32B"),
    ("MedGemma-4b-it",         "medgemma-4b-it"),
    ("MedGemma-27b-text-it",   "medgemma-27b-text-it"),
    ("GPT-5.4-nano",           "gpt-5.4-nano"),
    ("GPT-5.4-mini",           "gpt-5.4-mini"),
    ("GPT-5.5",                "gpt-5.5"),
]

PERSONA_AXES = ["personality", "health_literacy",
                "education_level", "past_medical_history_recall"]

# curated benchmark set + per-case persona profile + ICD chapter
curated = pd.read_json(PROJECT_ROOT / "data/processed/discharge_bench_curated_v3.jsonl",
                       lines=True)
CURATED_IDS = set(curated["note_id"])
profile_lookup = dict(zip(curated["note_id"], curated["profile"]))
icd_lookup     = dict(zip(curated["note_id"], curated["icd_chapter"]))
print(f"Curated cases: {len(CURATED_IDS)}")
# --- Shared labels + font, matching stratified_analysis.ipynb -----------
import matplotlib.font_manager as fm
for _ttf in (PROJECT_ROOT / "fonts").glob("*.ttf"):
    fm.fontManager.addfont(str(_ttf))
plt.rcParams["font.family"]     = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Fira Sans", "Nimbus Sans", "DejaVu Sans"]
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.labelsize"]  = 10

PERSONALITY_ABBR = {"neutral": "NEU", "high_conscientiousness": "HCO",
                    "minimiser": "MIN", "anxious": "ANX", "distrustful": "DIS"}
HEALTH_LIT_ABBR  = {"high": "HI", "low": "LO"}
RECALL_ABBR      = {"accurate": "ACC", "partial": "PAR", "poor": "POO"}
EDU_ABBR         = {"elementary": "EL", "high_school": "HI", "college": "CO"}

ICD_ABBR = {
    "Neoplasms":                                                    "NEO",
    "Diseases of the Circulatory System":                           "CIR",
    "Diseases of the Musculoskeletal System and Connective Tissue": "MSK",
    "Diseases of the Genitourinary System":                         "GEN",
    "Diseases of the Digestive System":                             "DIG",
    "Diseases of the Respiratory System":                           "RES",
    "Diseases of the Blood and Blood-Forming Organs":               "BLD",
    "Diseases of the Skin and Subcutaneous Tissue":                 "SKN",
    "Diseases of the Nervous System":                               "NER",
    "Infectious and Parasitic Diseases":                            "IPD",
    "Injury and Poisoning":                                         "INJ",
    "Endocrine, Nutritional and Metabolic Diseases":                "END",
    "Factors Influencing Health Status":                            "FHS",
    "Symptoms, Signs and Ill-Defined Conditions":                   "SSI",
    "Pregnancy, Childbirth and the Puerperium":                     "PCP",
    "Congenital Anomalies":                                         "CON",
}

AXIS_ORDER = {
    "personality":     ["neutral", "high_conscientiousness", "minimiser", "anxious", "distrustful"],
    "health_literacy": ["high", "low"],
    "education_level": ["elementary", "high_school", "college"],
    "past_medical_history_recall": ["accurate", "partial", "poor"],
    "icd_chapter":     sorted(ICD_ABBR.keys(), key=lambda k: ICD_ABBR[k]),
}
AXIS_LABELS = {
    "personality":     PERSONALITY_ABBR,
    "health_literacy": HEALTH_LIT_ABBR,
    "education_level": EDU_ABBR,
    "past_medical_history_recall": RECALL_ABBR,
    "icd_chapter":     ICD_ABBR,
}


In [ ]:
# --- Classify every case into one outcome -------------------------------

def classify(record):
    """Return (outcome, is_error, is_behavior_fail)."""
    err = record.get("error")
    if err:
        if "Connection error" in err:
            outcome = "error_connection"
        elif "maximum context length" in err:
            outcome = "error_context_length"
        elif "JSON" in err or "json" in err:
            outcome = "error_json"
        else:
            outcome = "error_other"
        return outcome, True, False
    if record.get("early_termination"):
        return "early_termination", False, True
    if not record.get("session_complete"):
        return "max_turns", False, True
    return "clean_complete", False, False


rows = []
RUNS = [f"run{i}" for i in range(1, 6)]   # 5 unseeded runs: 320 cases x 5 = 1,600 sessions per model
for display_name, model_dir in MODELS:
  for run in RUNS:
    path = OUTPUT_ROOT / f"{model_dir}__{run}" / "simulation_results.jsonl"
    with open(path) as f:
        records = [json.loads(line) for line in f if line.strip()]
    records = [r for r in records if r.get("note_id") in CURATED_IDS]
    ids = [r["note_id"] for r in records]
    if len(ids) != len(set(ids)) or set(ids) != CURATED_IDS:
        raise ValueError(f"{path}: {len(ids)} records, {len(set(ids))} unique, expected the 320 v3 cases once each")
    for r in records:
        outcome, is_err, is_beh = classify(r)
        prof = profile_lookup.get(r["note_id"], {})
        rows.append({
            "Model":             display_name,
            "run":               run,
            "note_id":           r["note_id"],
            "outcome":           outcome,
            "is_error":          is_err,
            "is_behavior_fail":  is_beh,
            "is_clean":          outcome == "clean_complete",
            "stop_reason":       r.get("stop_reason"),
            "n_accepted":        r.get("n_accepted"),
            "icd_chapter":       icd_lookup.get(r["note_id"]),
            **{ax: prof.get(ax) for ax in PERSONA_AXES},
        })

cases = pd.DataFrame(rows)
print(f"Classified {len(cases)} cases across {cases['Model'].nunique()} models")
cases.head()


In [ ]:
# --- Per-model outcome summary ------------------------------------------
OUTCOME_ORDER = ["clean_complete", "early_termination", "max_turns",
                 "error_connection", "error_context_length",
                 "error_json", "error_other"]
MODEL_ORDER = [d for d, _ in MODELS if d in set(cases["Model"])]

summary = (cases.groupby(["Model", "outcome"]).size()
                .unstack(fill_value=0)
                .reindex(index=MODEL_ORDER, columns=OUTCOME_ORDER, fill_value=0))
summary["total"] = summary.sum(axis=1)

summary_csv = OUT_DIR / "failure_summary.csv"
summary.to_csv(summary_csv)
print(f"Saved -> {summary_csv}")
summary

In [ ]:
# --- Artifact 1: per-model outcome distribution -------------------------
# Three-segment horizontal stacked bar per model:
#   clean complete          = session_complete
#   erroneous simulation    = early_termination + max_turns
#   system failure          = error_* (infrastructure: connection / context length / JSON / other)
PALETTE = {
    "clean complete":       "#3B5F91",
    "erroneous simulation": "#A15340",
    "system failure":       "#B0B0B0",
}

n_clean   = summary["clean_complete"].to_numpy()
n_erron   = (summary["early_termination"] + summary["max_turns"]).to_numpy()
n_sysfail = (summary["error_connection"] + summary["error_context_length"]
             + summary["error_json"] + summary["error_other"]).to_numpy()
n_total   = summary["total"].to_numpy()
pct_err   = n_erron   / n_total * 100
pct_sys   = n_sysfail / n_total * 100
y_pos     = np.arange(len(MODEL_ORDER))

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(y_pos, n_clean,   label="clean complete",
        color=PALETTE["clean complete"], edgecolor="0.3", linewidth=0.4)
ax.barh(y_pos, n_erron,   left=n_clean, label="erroneous simulation",
        color=PALETTE["erroneous simulation"], edgecolor="0.3", linewidth=0.4)
ax.barh(y_pos, n_sysfail, left=n_clean + n_erron, label="system failure",
        color=PALETTE["system failure"], edgecolor="0.3", linewidth=0.4)

# Annotate clean complete counts + percentage inside the blue segment.
pct_clean = n_clean / n_total * 100
for i, (c, p_c) in enumerate(zip(n_clean, pct_clean)):
    ax.text(c / 2, i, f"{c} ({p_c:.1f}%)", va="center", ha="center",
            color="white", fontsize=11, fontweight="normal")

# Annotate erroneous + system-failure counts to the right of each row.
x_max = n_total.max()
for i, (e, p_e, total) in enumerate(zip(n_erron, pct_err, n_total)):
    label = f"{e} ({p_e:.1f}%)"
    ax.text(total + x_max * 3 / 320, i, label, va="center", ha="left",
            color=PALETTE["erroneous simulation"], fontsize=11, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(MODEL_ORDER)
ax.invert_yaxis()
ax.set_xlabel("Number of sessions")
ax.set_xlim(0, x_max * 1.12)  # extra room for annotations
ax.set_title("Simulation outcome distribution per educator model", pad=28, fontsize=14, fontweight="bold")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=11, frameon=False)
fig.tight_layout()

f1 = FIG_DIR / "failure_outcome_distribution.png"
fig.savefig(f1, dpi=300, bbox_inches="tight")
print(f"Saved -> {f1}")
plt.show()


In [ ]:
# --- Artifact 2: model x persona Incomplete-Simulation Rate heatmap grid ---
# 2x2 panels for the four patient-profile axes, sharing the y-axis (models)
# and a single colorbar.
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def rate_table(flag_col, axis):
    t = (cases.groupby(["Model", axis])[flag_col].mean()
               .unstack().reindex(index=MODEL_ORDER))
    levels = [lv for lv in AXIS_ORDER[axis] if lv in t.columns]
    t = t[levels]
    t.columns = [AXIS_LABELS[axis].get(lv, lv) for lv in levels]
    return t

PANEL_ORDER = ["personality", "health_literacy",
               "education_level", "past_medical_history_recall"]

fig, axes = plt.subplots(2, 2, sharey=True, figsize=(10, 8))
for ax, axis in zip(axes.flat, PANEL_ORDER):
    tbl = rate_table("is_behavior_fail", axis)
    sns.heatmap(tbl, annot=True, fmt=".2f", cmap="cividis",
                vmin=0, vmax=0.3, ax=ax, cbar=False,
                linewidths=0.5, linecolor="white")
    title = axis.replace("_", " ").title()
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")

# Single shared colorbar across the grid.
sm   = ScalarMappable(cmap="cividis", norm=Normalize(vmin=0, vmax=0.3))
cbar = fig.colorbar(sm, ax=axes, shrink=0.5,
                    label="Incomplete Simulation Rate")
fig.suptitle("Incomplete Simulation Rate by Patient Profile", y=1.02)

fpath = FIG_DIR / "failure_heatmap_grid.png"
fig.savefig(fpath, dpi=300, bbox_inches="tight")
print(f"Saved -> {fpath}")
plt.show()


In [ ]:
# --- Artifact 2b: model x ICD chapter Incomplete-Simulation Rate heatmap ---
# Single panel — too many levels (≤24) for a 2x2 grid. Drops chapters with
# fewer than MIN_N total cases across all models so noisy cells don't dominate.
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

MIN_N = 10  # min total cases per chapter to include

# Per (Model, chapter) failure rate + counts for filtering.
icd_n    = cases.groupby("icd_chapter").size()
keep_lvl = [lv for lv in AXIS_ORDER["icd_chapter"]
            if lv in icd_n.index and icd_n[lv] >= MIN_N]
print(f"Including {len(keep_lvl)} of {len(AXIS_ORDER['icd_chapter'])} chapters "
      f"(min n = {MIN_N}); dropped {len(AXIS_ORDER['icd_chapter']) - len(keep_lvl)}.")

icd_tbl = (cases.groupby(["Model", "icd_chapter"])["is_behavior_fail"].mean()
                .unstack().reindex(index=MODEL_ORDER)[keep_lvl])
icd_tbl.columns = [ICD_ABBR[c] for c in icd_tbl.columns]

fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(keep_lvl)), 5))
sns.heatmap(icd_tbl, annot=True, fmt=".2f", cmap="cividis",
            vmin=0, vmax=0.3, ax=ax, cbar=False,
            linewidths=0.5, linecolor="white")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Incomplete Simulation Rate by ICD Chapter")

sm   = ScalarMappable(cmap="cividis", norm=Normalize(vmin=0, vmax=0.3))
fig.colorbar(sm, ax=ax, shrink=0.85, label="Incomplete Simulation Rate")

fpath = FIG_DIR / "failure_heatmap_icd_chapter.png"
fig.savefig(fpath, dpi=300, bbox_inches="tight")
print(f"Saved -> {fpath}")
plt.show()


In [ ]:
# --- Artifact 3: candidate transcripts for qualitative case studies -----
# Behavior failures only (early_termination + max_turns) -- these are the
# model-attributable cases worth reading. Pick 2-3 from this list.

candidates = (cases[cases["is_behavior_fail"]]
              [["Model", "note_id", "outcome", "n_accepted",
                "personality", "health_literacy", "stop_reason"]]
              .sort_values(["outcome", "Model"])
              .reset_index(drop=True))

cand_csv = OUT_DIR / "failure_case_candidates.csv"
candidates.to_csv(cand_csv, index=False)
print(f"{len(candidates)} behavior-failure cases  ->  {cand_csv}\n")

# preview: outcome counts + a few stop_reason texts per outcome
print(candidates["outcome"].value_counts(), "\n")
for outcome in candidates["outcome"].unique():
    sub = candidates[candidates["outcome"] == outcome].head(3)
    print(f"=== {outcome} (showing 3) ===")
    for _, r in sub.iterrows():
        sr = (r["stop_reason"] or "")[:240]
        print(f"  [{r['Model']}] {r['note_id']}  persona={r['personality']}/{r['health_literacy']}")
        print(f"    {sr}")
    print()